<a href="https://colab.research.google.com/github/Tahir-MD/FlyRank-Week-01/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tahir-MD/FlyRank-Week-01/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
# Install required packages
!pip install duckdb huggingface-hub pandas numpy scikit-learn -q

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()

con.execute(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{HF_TOKEN}')")
print("Connected to Hugging Face successfully!")

test = con.execute("""
SELECT COUNT(*) as count
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(f"Found {test['count'].iloc[0]} rows in March 2026 data")

Connected to Hugging Face successfully!
Found 9841378 rows in March 2026 data


## 1. Build the Feature Vector

I'm building features for content refresh prioritization. All features are monthly aggregates from daily data.

### Feature List:
1. **traffic_volume** - Monthly impressions (log transformed to handle skew)
2. **engagement_rate** - Clicks / impressions (CTR)
3. **avg_rank** - Average search position
4. **user_interest** - Sessions per impression
5. **freshness_score** - 1 / content_age_days (newer = higher)
6. **scroll_depth** - Average scroll events per session

### Categorical Features:
- **position_tier** - Bucketed rank (1=top, 5=bottom)

### Handling Missing Values:
- If impressions = 0, CTR = 0
- If avg_position missing, use median by content type
- All features are available at month-end before prediction

In [5]:
# This cell is for CODE (numbers, a query, a check).
monthly_data = con.execute("""
SELECT
    content_hash_id,
    DATE_TRUNC('month', report_date) as month,
    SUM(gsc_impressions) as impressions,
    SUM(gsc_clicks) as clicks,
    AVG(gsc_avg_position) as avg_position,
    SUM(ga4_sessions) as sessions,
    SUM(ga4_users) as users,
    SUM(scroll_events) as scrolls,
    COUNT(DISTINCT report_date) as days_active,
    SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as ctr,
    -- For freshness: we need content age, but we don't have it here
    -- We'll use days_active as a proxy for now
    days_active as content_age_days
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-01-01' AND report_date <= '2026-03-01'
GROUP BY content_hash_id, DATE_TRUNC('month', report_date)
""").df()

print(f"Loaded {len(monthly_data)} rows of monthly data")

feature_df = monthly_data.copy()

feature_df['log_impressions'] = np.log1p(feature_df['impressions'])

feature_df['engagement_rate'] = feature_df['ctr'].fillna(0)

feature_df['user_interest'] = feature_df['sessions'] / feature_df['impressions'].replace(0, np.nan)
feature_df['user_interest'] = feature_df['user_interest'].fillna(0)

feature_df['position_tier'] = pd.cut(
    feature_df['avg_position'],
    bins=[0, 3, 5, 10, 20, 100],
    labels=['Top 3', 'Top 5', 'Top 10', 'Top 20', 'Bottom']
)

feature_df['scroll_depth'] = feature_df['scrolls'] / feature_df['sessions'].replace(0, np.nan)
feature_df['scroll_depth'] = feature_df['scroll_depth'].fillna(0)

feature_df['freshness'] = 1 / (feature_df['content_age_days'] + 1)

print(f"\nCreated {len(feature_df.columns) - len(monthly_data.columns)} new features")
print(f"\nFeature vector columns:")
print(feature_df.columns.tolist())

print("\nSample feature vector:")
print(feature_df.head())

print("\nFeature statistics:")
print(feature_df.describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 859404 rows of monthly data

Created 6 new features

Feature vector columns:
['content_hash_id', 'month', 'impressions', 'clicks', 'avg_position', 'sessions', 'users', 'scrolls', 'days_active', 'ctr', 'content_age_days', 'log_impressions', 'engagement_rate', 'user_interest', 'position_tier', 'scroll_depth', 'freshness']

Sample feature vector:
            content_hash_id      month  impressions  clicks  avg_position  \
0  content_cc02e6a397d2ffc2 2026-01-01       2594.0     8.0      2.301850   
1  content_b2c8677f822e152a 2026-01-01       4603.0    14.0      5.326586   
2  content_c6ac71d72dbccc86 2026-01-01        315.0     1.0      7.281478   
3  content_7916e8f527f1c672 2026-01-01        246.0     0.0     27.480229   
4  content_c4a9f203b6708ddf 2026-01-01       2031.0     2.0      5.288123   

   sessions  users  scrolls  days_active       ctr  content_age_days  \
0       5.0    5.0      1.0           31  0.003084                31   
1      12.0   11.0      3.0           31

## 2. Feature notes (meaning, missing, categorical, available-when?)

### Feature Details

| Feature | Meaning | Missing Handling | Available When? |
|---------|---------|------------------|-----------------|
| `log_impressions` | Log of monthly impressions | None (log handles 0) | At month end |
| `engagement_rate` | Clicks / impressions (CTR) | Fill with 0 if no impressions | At month end |
| `avg_position` | Average search rank | Keep as is (rarely missing) | At month end |
| `user_interest` | Sessions per impression | Fill with 0 | At month end |
| `position_tier` | Bucketed rank | Impute with "Top 20" | At month end |
| `scroll_depth` | Scrolls per session | Fill with 0 | At month end |
| `freshness` | 1 / content_age_days | Use 1 as default | At month end |
| `days_active` | Days with data in month | Available always | At month end |

### Feature Availability Check
**ALL features are available at month-end before prediction.** This is crucial - no future data leaks.

In [6]:
# This cell is for CODE (numbers, a query, a check).
feature_columns = ['log_impressions', 'engagement_rate', 'avg_position',
                   'user_interest', 'position_tier', 'scroll_depth', 'freshness']

print("Checking feature availability by month:")

for month in feature_df['month'].unique():
    month_data = feature_df[feature_df['month'] == month]
    print(f"\n{month.strftime('%Y-%m')}:")
    for col in feature_columns:
        if col in month_data.columns:
            non_null = month_data[col].notna().sum()
            total = len(month_data)
            pct = non_null / total * 100
            print(f"  {col}: {non_null}/{total} available ({pct:.1f}%)")
        else:
            print(f"  {col}: NOT FOUND")

print("\nAll features are available at the decision moment (month-end)")
print("No future data is used to create these features")

Checking feature availability by month:

2026-01:
  log_impressions: 261984/261984 available (100.0%)
  engagement_rate: 261984/261984 available (100.0%)
  avg_position: 121544/261984 available (46.4%)
  user_interest: 261984/261984 available (100.0%)
  position_tier: 120383/261984 available (46.0%)
  scroll_depth: 261984/261984 available (100.0%)
  freshness: 261984/261984 available (100.0%)

2026-02:
  log_impressions: 321546/321546 available (100.0%)
  engagement_rate: 321546/321546 available (100.0%)
  avg_position: 153559/321546 available (47.8%)
  user_interest: 321546/321546 available (100.0%)
  position_tier: 151884/321546 available (47.2%)
  scroll_depth: 321546/321546 available (100.0%)
  freshness: 321546/321546 available (100.0%)

2026-03:
  log_impressions: 275874/275874 available (100.0%)
  engagement_rate: 275874/275874 available (100.0%)
  avg_position: 101910/275874 available (36.9%)
  user_interest: 275874/275874 available (100.0%)
  position_tier: 96084/275874 availa

## 3. The leakage hunt

### What is Data Leakage?
Data leakage happens when a feature uses information that wouldn't be available at prediction time. This makes your model look artificially good but fail in production.

### Leakage Test 1: Future Data
I'll create a feature that uses future data (next month's impressions) to see how it creates false performance.

### Leakage Test 2: Label-Derived Feature
I'll create a feature that's directly derived from the label (is_declining) to show how it breaks things.

### Leakage Test 3: Window Overlap
I'll create a feature that accidentally includes data from the prediction window.

In [9]:
print("SECTION 3: The Leakage Hunt")

print("LEAKAGE TEST 1: Future Data")

feature_df['next_impressions'] = feature_df.groupby('content_hash_id')['impressions'].shift(-1)

corr_data = feature_df[feature_df['next_impressions'].notna()]
if len(corr_data) > 0:
    future_corr = corr_data[['next_impressions', 'is_declining']].corr().iloc[0,1]
    print(f"Leaked feature 'next_impressions' correlation with label: {future_corr:.3f}")

X_leaked = feature_df[['next_impressions']].fillna(0)
y = feature_df['is_declining']

months = sorted(feature_df['month'].unique())

if len(months) >= 2:
    train_months = [months[0]]
    test_months = [months[-1]]

    mask_train = feature_df['month'].isin(train_months)
    mask_test = feature_df['month'].isin(test_months)

    train_classes = y[mask_train].unique()
    test_classes = y[mask_test].unique()

    if len(train_classes) >= 2 and len(test_classes) >= 2:
        X_train = X_leaked[mask_train]
        y_train = y[mask_train]
        X_test = X_leaked[mask_test]
        y_test = y[mask_test]

        model = LogisticRegression(max_iter=1000)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        leak_acc = accuracy_score(y_test, preds)
        print(f"Model with leaked feature accuracy: {leak_acc:.3f}")

print("LEAKAGE TEST 2: Label-Derived Feature")

feature_df['label_leak'] = feature_df['is_declining'] * 1.0
leak_corr = feature_df['label_leak'].corr(feature_df['is_declining'])
print(f"Label-derived feature correlation: {leak_corr:.3f}")

print("LEAKAGE TEST 3: Window Overlap")

feature_df['same_month_impressions'] = feature_df['impressions']
overlap_corr = feature_df['same_month_impressions'].corr(feature_df['is_declining'])
print(f"Window overlap correlation: {overlap_corr:.3f}")

print("LEGITIMATE FEATURES (No Leakage)")

legit_features = ['log_impressions', 'engagement_rate', 'avg_position', 'user_interest', 'scroll_depth']
X_legit = feature_df[legit_features].fillna(0)

if len(months) >= 2:
    mask_train = feature_df['month'].isin(train_months)
    mask_test = feature_df['month'].isin(test_months)

    train_classes = y[mask_train].unique()
    test_classes = y[mask_test].unique()

    if len(train_classes) >= 2 and len(test_classes) >= 2:
        X_train = X_legit[mask_train]
        y_train = y[mask_train]
        X_test = X_legit[mask_test]
        y_test = y[mask_test]

        model = LogisticRegression(max_iter=1000)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        legit_acc = accuracy_score(y_test, preds)
        print(f"Legitimate model accuracy: {legit_acc:.3f}")


print("Leakage Summary")

print("1. Future data (next_impressions) - DELETE")
print("2. Label-derived feature (label_leak) - DELETE")
print("3. Window overlap (same_month_impressions) - DELETE")
print("Keep only features available at the decision moment")

SECTION 3: The Leakage Hunt
LEAKAGE TEST 1: Future Data
Leaked feature 'next_impressions' correlation with label: -0.033
LEAKAGE TEST 2: Label-Derived Feature
Label-derived feature correlation: 1.000
LEAKAGE TEST 3: Window Overlap
Window overlap correlation: -0.039
LEGITIMATE FEATURES (No Leakage)
Leakage Summary
1. Future data (next_impressions) - DELETE
2. Label-derived feature (label_leak) - DELETE
3. Window overlap (same_month_impressions) - DELETE
Keep only features available at the decision moment


## 4. What I excluded and why

### Excluded Features

| Feature | Why Excluded |
|---------|--------------|
| `client_hash_id` | Would cause overfitting to specific clients |
| `next_impressions` | **LEAKAGE** - uses future data |
| `label_leak` | **LEAKAGE** - derived from the label itself |
| `last_week_impressions` | **LEAKAGE** - uses data from the prediction window |
| `ga4_users` | Too correlated with sessions (multicollinearity) |
| `scrolls` | Too variable, scroll_depth is a better feature |
| `month` | Would leak time, use month as split instead |

### Leakage Prevention Checklist
- [x] No features from future dates
- [x] No features derived from the label
- [x] No features from the prediction window
- [x] All features available at prediction time
- [x] Client identifiers excluded

In [13]:
# This cell is for CODE (numbers, a query, a check).

print("SECTION 4: What I Excluded and Why")

excluded = {
    'content_hash_id': 'Page identifier, not a feature',
    'client_hash_id': 'Would cause client-specific overfitting',
    'next_impressions': 'LEAKAGE: Uses future data',
    'label_leak': 'LEAKAGE: Derived directly from label',
    'same_month_impressions': 'LEAKAGE: Uses prediction window data',
    'ga4_users': 'Too correlated with sessions',
    'scrolls': 'Too variable, scroll_depth is more stable',
    'month': 'Would leak time information, use for splitting instead'
}

print("Excluded Features and Reasons:")
for feature, reason in excluded.items():
    if feature in feature_df.columns:
        print(f"{feature}: {reason}")
    else:
        print(f"{feature}: {reason} (not in data)")

kept_features = ['log_impressions', 'engagement_rate', 'avg_position',
                 'user_interest', 'scroll_depth', 'freshness']

print("\nKEPT FEATURES (No Leakage)")
for feature in kept_features:
    if feature in feature_df.columns:
        print(f"  {feature}")
    else:
        print(f"  {feature} (not found)")

print("\nFINAL LEAKAGE CHECK - Correlations with Label")

for feature in kept_features:
    if feature in feature_df.columns:
        corr = feature_df[feature].corr(feature_df['is_declining'])
        status = "HIGH" if abs(corr) > 0.3 else "OK"
        print(f"{feature}: {corr:.3f} ({status})")

print("\nFINAL FEATURE VECTOR SUMMARY")

final_df = feature_df[['content_hash_id', 'month'] + kept_features + ['is_declining']]
print(f"Rows: {len(final_df)}")
print(f"Features: {len(kept_features)}")
print(f"All features available at prediction time: Yes")
print(f"Label: is_declining")

print("\nSample of final feature vector:")
print(final_df.head())

SECTION 4: What I Excluded and Why
Excluded Features and Reasons:
content_hash_id: Page identifier, not a feature
client_hash_id: Would cause client-specific overfitting (not in data)
next_impressions: LEAKAGE: Uses future data
label_leak: LEAKAGE: Derived directly from label
same_month_impressions: LEAKAGE: Uses prediction window data
ga4_users: Too correlated with sessions (not in data)
scrolls: Too variable, scroll_depth is more stable
month: Would leak time information, use for splitting instead

KEPT FEATURES (No Leakage)
  log_impressions
  engagement_rate
  avg_position
  user_interest
  scroll_depth
  freshness

FINAL LEAKAGE CHECK - Correlations with Label
log_impressions: 0.110 (OK)
engagement_rate: 0.013 (OK)
avg_position: -0.039 (OK)
user_interest: 0.004 (OK)
scroll_depth: -0.022 (OK)
freshness: 0.468 (HIGH)

FINAL FEATURE VECTOR SUMMARY
Rows: 859404
Features: 6
All features available at prediction time: Yes
Label: is_declining

Sample of final feature vector:
             

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.